# 03.04 · CNNs & RNNs: what came before transformers

**Chapter:** Deep Learning · **Milestone:** M1

## Goals
- [ ] Convolutions: local patterns, weight sharing, far fewer parameters than an MLP
- [ ] Train a small CNN and beat the MLP from 03.03
- [ ] RNNs: processing sequences step by step, and why vanishing gradients + no parallelism led to transformers

## Resources
- [Understanding Deep Learning (free book), ch. 10 & 12](https://udlbook.github.io/udlbook/) — free
- [CS231n – Convolutional networks notes](https://cs231n.github.io/convolutional-networks/) — free
- [Karpathy – The Unreasonable Effectiveness of RNNs](https://karpathy.github.io/2015/05/21/rnn-effectiveness/) — free

You won't build many CNNs/RNNs as an AI engineer, but vision encoders inside multimodal LLMs, speech models and some
"state-space" LLM alternatives (Mamba) descend from these ideas.

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from aieng import DATA

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)

## 1. A convolution is a sliding dot product
A small kernel (e.g. 3×3) slides over the image; each output pixel is the dot product of the kernel with the patch under it.
Hand-made kernels below detect edges. A CNN **learns** its kernels.

In [ ]:
tfm = transforms.ToTensor()
train_set = datasets.FashionMNIST(DATA, train=True, download=True, transform=transforms.Compose([tfm, transforms.Normalize((0.286,), (0.353,))]))
test_set = datasets.FashionMNIST(DATA, train=False, download=True, transform=transforms.Compose([tfm, transforms.Normalize((0.286,), (0.353,))]))
img = train_set[0][0].unsqueeze(0)  # (1, 1, 28, 28)

kernels = {
    "vertical edges": torch.tensor([[-1.0, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "horizontal edges": torch.tensor([[-1.0, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "blur": torch.ones(3, 3) / 9,
}
fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
axes[0].imshow(img[0, 0], cmap="gray"); axes[0].set_title("input")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    ax.imshow(nn.functional.conv2d(img, k.view(1, 1, 3, 3), padding=1)[0, 0], cmap="gray"); ax.set_title(name)
for ax in axes: ax.axis("off")
plt.show()

## 2. A small CNN
Conv → activation → pooling (downsample), twice, then a linear classifier. Weight sharing: the same 3×3 kernel is used at every
position, so a conv layer has few parameters no matter the image size.

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.GELU(), nn.MaxPool2d(2),  # 28 -> 14
    nn.Conv2d(32, 64, 3, padding=1), nn.GELU(), nn.MaxPool2d(2),  # 14 -> 7
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.GELU(), nn.Linear(128, 10),
).to(device)
conv_params = sum(p.numel() for p in cnn[:6].parameters())
print(f"conv layers: {conv_params:,} params | whole CNN: {sum(p.numel() for p in cnn.parameters()):,} | 03.03 MLP: ~535,000")

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=1024)
opt = torch.optim.AdamW(cnn.parameters(), lr=1e-3)

EPOCHS = 2
for epoch in range(EPOCHS):
    cnn.train(); start = time.perf_counter()
    for xb, yb in train_loader:
        loss = nn.functional.cross_entropy(cnn(xb.to(device)), yb.to(device))
        opt.zero_grad(); loss.backward(); opt.step()
    cnn.eval()
    with torch.no_grad():
        acc = sum((cnn(xb.to(device)).argmax(1).cpu() == yb).sum().item() for xb, yb in test_loader) / len(test_set)
    print(f"epoch {epoch}: test acc {acc:.3f} ({time.perf_counter() - start:.0f}s)")

Learned first-layer kernels:

In [ ]:
w = cnn[0].weight.detach().cpu()[:16, 0]
fig, axes = plt.subplots(2, 8, figsize=(8, 2))
for ax, k in zip(axes.ravel(), w):
    ax.imshow(k, cmap="coolwarm"); ax.axis("off")
plt.show()

## 3. RNNs: one step at a time
An RNN keeps a hidden state $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$ and reads the sequence token by token.
Two fundamental problems:
1. **Sequential**: step $t$ needs step $t-1$, so you can't parallelise over the sequence on a GPU.
2. **Vanishing gradients**: the gradient from the loss to early tokens is multiplied by $W_h$ (and tanh') once per step.

Demo: measure how much the final hidden state depends on each input position.

In [ ]:
torch.manual_seed(0)
rnn = nn.RNN(input_size=8, hidden_size=64, batch_first=True)
x = torch.randn(1, 100, 8, requires_grad=True)
out, h = rnn(x)
h.sum().backward()
grad_per_step = x.grad[0].norm(dim=1)

plt.semilogy(grad_per_step); plt.xlabel("input position (0 = first token)"); plt.ylabel("‖∂h_final / ∂x_t‖")
plt.title("vanilla RNN: early tokens barely influence the final state"); plt.show()

LSTMs/GRUs add gates to fight this, and worked well until 2017. **Attention** (next chapter) solves both problems at once:
every token looks directly at every other token (no long chain), and all positions are computed in parallel.

## Exercises
1. Run the gradient demo with `nn.LSTM` instead (use `out[:, -1]` as the final state). How does the curve change?
2. Train the CNN for 5 epochs with `nn.Dropout(0.3)` before the last layer. Can you beat 91%?
3. Download CIFAR-10 (`datasets.CIFAR10`, ~170 MB, colour images) and adapt the CNN (3 input channels, 32×32). What accuracy do you get?

In [ ]:
# your code here